# NB_hospital_daily_summary

Builds the **`databricks_rg.gold.hospital_daily_summary`** gold table — one row per hospital per calendar day with test-volume counts, abnormal/critical rates, and day-over-day trend via LAG(). Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_rg.gold.hospital_daily_summary AS

WITH base AS (
    SELECT
        lr.hospital_id,
        h.hospital_name,
        lr.test_date                                                     AS summary_date,
        c.day_name,
        c.is_weekend,
        COUNT(*)                                                         AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                    AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                     AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)       AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)       AS critical_test_count
    FROM databricks_rg.silver.lab_results      lr
    LEFT JOIN databricks_rg.silver.hospitals   h
           ON lr.hospital_id = h.hospital_id
    LEFT JOIN databricks_rg.silver.calendar    c
           ON c.date_key = CAST(DATE_FORMAT(lr.test_date, 'yyyyMMdd') AS INT)
    GROUP BY lr.hospital_id, h.hospital_name, lr.test_date, c.day_name, c.is_weekend
)

SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    ROUND(abnormal_test_count * 100.0 / total_tests, 2)                  AS abnormal_rate_pct,
    ROUND(critical_test_count * 100.0 / total_tests, 2)                  AS critical_rate_pct,
    LAG(total_tests) OVER (
        PARTITION BY hospital_id ORDER BY summary_date
    )                                                                    AS prev_day_total_tests,
    ROUND(
        (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
        * 100.0
        / NULLIF(LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date), 0)
    , 2)                                                                 AS test_volume_change_pct
FROM base

In [0]:
df_hospital_daily = spark.table("databricks_rg.gold.hospital_daily_summary")
record_count = df_hospital_daily.count()
dbutils.notebook.exit(str(record_count))